# HDB Resale Flat Prices — ETL Pipeline
**Senior Data Engineer Technical Test — Part 1**

This notebook builds an end-to-end ETL pipeline over the *Resale Flat Prices*
dataset (Jan 2012 – Dec 2016) published at
[data.gov.sg/collections/189](https://data.gov.sg/collections/189/view), covering:

1. Extraction & combination of source files into a single master dataset
2. Data profiling
3. Data validation (Date, Town, Flat Type, Flat Model, Storey Range)
4. Remaining lease recomputation (99-year lease, as of today)
5. Composite-key deduplication (higher price wins)
6. Price anomaly detection (flagged, documented heuristic)
7. Transformation: `Resale Identifier` construction
8. Irreversible hashing of the identifier (SHA-256)
9. Writing the 5 mandated output groups: **raw / cleaned / transformed / failed / hashed**

> **A note on data source in this sandbox:** the environment this notebook was
> authored in has no outbound internet access, so a synthetic dataset that is
> **schema-identical** to the real data.gov.sg files (`month, town, flat_type,
> block, street_name, storey_range, floor_area_sqm, flat_model,
> lease_commence_date, resale_price`) is generated locally by
> `src/generate_sample_data.py` and used to demonstrate the pipeline end-to-end,
> including deliberately injected duplicates, invalid enum values, nulls and
> price outliers so every data-quality rule below has something to catch.
> **To run this notebook against the real data**, replace the call in
> **Section 1** with `download_from_data_gov_sg()` (implemented below using the
> data.gov.sg Datastore API) — no other cell needs to change, since everything
> downstream depends only on the (identical) column schema.


In [1]:
import sys, os, glob
sys.path.append(os.path.abspath("src"))

import pandas as pd
import numpy as np

import pipeline as pl   # local module: src/pipeline.py — all core ETL functions

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

RAW_DIR, CLEANED_DIR = "data/raw", "data/cleaned"
TRANSFORMED_DIR, FAILED_DIR, HASHED_DIR = "data/transformed", "data/failed", "data/hashed"
for d in [RAW_DIR, CLEANED_DIR, TRANSFORMED_DIR, FAILED_DIR, HASHED_DIR]:
    os.makedirs(d, exist_ok=True)


## Section 1 — Extraction

`download_from_data_gov_sg()` below shows how the real files would be pulled
programmatically via the data.gov.sg Datastore API (no manual downloads, per
the assignment's "no manual interface interactions" requirement). It is not
invoked in this offline run — see the note above.


In [2]:
def download_from_data_gov_sg(dataset_ids: dict, out_dir: str = RAW_DIR) -> list[str]:
    """
    Programmatic download via the data.gov.sg Datastore API — no manual portal
    clicks. `dataset_ids` maps a friendly filename to the dataset's resource id
    (found on the dataset's page on data.gov.sg). Paginates through
    `datastore_search` until all records for that resource are retrieved.
    Not executed in this sandbox (no outbound network access) — kept here so
    the notebook is drop-in runnable in an environment that has internet access.
    """
    import requests

    paths = []
    base = "https://api-open.data.gov.sg/v1/public/api/datasets/{}/poll-download"
    for filename, resource_id in dataset_ids.items():
        resp = requests.get(base.format(resource_id), timeout=30).json()
        download_url = resp["data"]["url"]
        csv_bytes = requests.get(download_url, timeout=60).content
        out_path = os.path.join(out_dir, filename)
        with open(out_path, "wb") as f:
            f.write(csv_bytes)
        paths.append(out_path)
    return paths


# Example (resource IDs are illustrative — confirm the current ones on the
# dataset's data.gov.sg page before use):
# RAW_PATHS = download_from_data_gov_sg({
#     "resale-flat-prices-2012-2014.csv": "<resource_id_2012_2014>",
#     "resale-flat-prices-2015-2016.csv": "<resource_id_2015_2016>",
# })


In [3]:
# Offline sandbox: generate a schema-identical synthetic dataset instead
# (see src/generate_sample_data.py for full docstring/rationale).
import generate_sample_data  # executes __main__ guard is False on import; call explicitly below

months_2012_2014 = generate_sample_data.make_month_range("2012-01", "2014-12")
months_2015_2016 = generate_sample_data.make_month_range("2015-01", "2016-12")

df1 = generate_sample_data.generate_file(months_2012_2014, 6000, seed=1)
df2 = generate_sample_data.generate_file(months_2015_2016, 4500, seed=2)

df1.to_csv(f"{RAW_DIR}/resale-flat-prices-based-on-registration-date-from-jan-2012-to-dec-2014.csv", index=False)
df2.to_csv(f"{RAW_DIR}/resale-flat-prices-based-on-registration-date-from-jan-2015-to-dec-2016.csv", index=False)

RAW_PATHS = sorted(glob.glob(f"{RAW_DIR}/*.csv"))
print(f"Raw files ready: {[os.path.basename(p) for p in RAW_PATHS]}")


Raw files ready: ['resale-flat-prices-based-on-registration-date-from-jan-2012-to-dec-2014.csv', 'resale-flat-prices-based-on-registration-date-from-jan-2015-to-dec-2016.csv']


In [4]:
master = pl.load_and_combine(RAW_PATHS)

# explicit typing (kept separate from load_and_combine so extraction stays a
# pure "as-is" structural union, per the requirement not to modify the raw data)
master["floor_area_sqm"] = pd.to_numeric(master["floor_area_sqm"], errors="coerce")
master["lease_commence_date"] = pd.to_numeric(master["lease_commence_date"], errors="coerce")
master["resale_price"] = pd.to_numeric(master["resale_price"], errors="coerce")

print(f"Master dataset: {master.shape[0]:,} rows x {master.shape[1]} columns")
master.head()


Master dataset: 10,552 rows x 11 columns
     month           town flat_type block          street_name storey_range  floor_area_sqm     flat_model  lease_commence_date  resale_price                                                                 _source_file
0  2013-12        GEYLANG    3 ROOM   235           Geylang DR     16 TO 18            82.5       Model A2                 1967      289000.0  resale-flat-prices-based-on-registration-date-from-jan-2012-to-dec-2014.csv
1  2012-07        PUNGGOL    3 ROOM  395B        Punggol LOR 1     04 TO 06            90.3        Model A                 1987      281000.0  resale-flat-prices-based-on-registration-date-from-jan-2012-to-dec-2014.csv
2  2012-04  BUKIT PANJANG    2 ROOM   118  Bukit Panjang AVE 3     28 TO 30           109.2           DBSS                 1968      180500.0  resale-flat-prices-based-on-registration-date-from-jan-2012-to-dec-2014.csv
3  2014-06    JURONG EAST    1 ROOM   768    Jurong East LOR 1     07 TO 09        

## Section 2 — Data Profiling

Custom, dependency-free profiler (row count, null %, distinct count,
min/max/mean for numeric columns, top values for categoricals). Functionally
equivalent to what `ydata-profiling` / `great_expectations` produce, without
adding a heavyweight dependency to the notebook.


In [5]:
profile = pl.profile_dataset(master)
profile


                 column    dtype  n_rows  n_null  pct_null  n_distinct                                                                                                                                                                  top_values            min           max       mean        std
0                 month      str   10552       0       0.0          61                                                                                                                            {'2016-11': 216, '2015-03': 208, '2016-12': 208}            NaN           NaN        NaN        NaN
1                  town      str   10552       0       0.0          27                                                                                                                {'BUKIT PANJANG': 450, 'BUKIT TIMAH': 435, 'TOA PAYOH': 430}            NaN           NaN        NaN        NaN
2             flat_type      str   10552       0       0.0           8                                                

## Section 3 — Data Validation

Validation rules for **Date (month), Town, Flat Type, Flat Model, storey_range**
are derived from the *statistical properties of the master dataset itself*
(the set of values observed with sufficient support / frequency), rather than
hardcoding Singapore's 26 towns etc. — see `pipeline.validate_dataset` docstring
for the exact thresholds and rationale for each field. Rows that fail any rule
go to `failed`, tagged with `_fail_reason`.


In [6]:
valid, failed_validation = pl.validate_dataset(master)
print(f"Valid: {len(valid):,} rows | Failed validation: {len(failed_validation):,} rows")
failed_validation["_fail_reason"].value_counts()


Valid: 10,514 rows | Failed validation: 38 rows
_fail_reason
invalid_flat_model;      21
invalid_storey_range;    13
invalid_month;            4


## Section 4 — Remaining Lease

HDB leases are assumed to be 99 years from `lease_commence_date` (1 Jan of
that year). Remaining lease = (lease_commence_date + 99y) − today, rounded
**down** to completed years + months.


In [7]:
valid = pl.compute_remaining_lease(valid)
valid[["town", "lease_commence_date", "remaining_lease_years", "remaining_lease_months", "remaining_lease"]].head()


            town  lease_commence_date  remaining_lease_years  remaining_lease_months    remaining_lease
0        GEYLANG                 1967                     39                       5  39 years 5 months
1        PUNGGOL                 1987                     59                       5  59 years 5 months
2  BUKIT PANJANG                 1968                     40                       5  40 years 5 months
3    JURONG EAST                 2001                     73                       5  73 years 5 months
4       CLEMENTI                 1977                     49                       5  49 years 5 months


## Section 5 — Deduplication (composite key = all columns except `resale_price`)

Where the composite key repeats, the **higher** `resale_price` is kept; the
lower-price duplicate is written to `failed`.


In [8]:
BUSINESS_COLS = [
    "month", "town", "flat_type", "block", "street_name", "storey_range",
    "floor_area_sqm", "flat_model", "lease_commence_date",
]
valid_dedup, failed_dup = pl.dedup_keep_higher_price(valid, BUSINESS_COLS)
print(f"Kept: {len(valid_dedup):,} | Discarded as lower-price duplicates: {len(failed_dup):,}")


Kept: 10,462 | Discarded as lower-price duplicates: 52


## Section 6 — Price Anomaly Detection (heuristic)

**Heuristic (documented):** robust z-score of `resale_price` within each
`(town, flat_type)` peer group, using **median / MAD** (median absolute
deviation, scaled by 1.4826 to be a consistent estimator of std under
normality) instead of mean/std, because resale prices are right-skewed and a
few extreme values would otherwise distort the very statistic used to judge
them. `|z| > 3` ⇒ flagged. Groups with fewer than 5 rows fall back to the
whole-dataset median/MAD to avoid unstable statistics on sparse groups.
Flagged rows are **kept, not discarded** — an anomaly is a candidate for
review, not proof of a data error.


In [9]:
cleaned = pl.flag_price_anomalies(valid_dedup)
n_anom = int(cleaned["_is_price_anomaly"].sum())
print(f"{n_anom} rows flagged as price anomalies out of {len(cleaned):,} ({n_anom/len(cleaned):.2%})")
cleaned.loc[cleaned["_is_price_anomaly"], ["town", "flat_type", "resale_price", "_price_zscore"]].head()


82 rows flagged as price anomalies out of 10,462 (0.78%)
            town  flat_type  resale_price  _price_zscore
35   BUKIT TIMAH     4 ROOM  5.540000e+05       3.022718
225  JURONG WEST     1 ROOM  1.670000e+05       6.610009
337   ANG MO KIO     3 ROOM  3.725000e+05       3.001959
353  BUKIT BATOK     3 ROOM  1.595536e+06      45.157951
379     TAMPINES  EXECUTIVE  6.190000e+05       3.399984


### Write CLEANED output
`cleaned` = dataset that has passed all data-quality requirements above
(validation + dedup), with anomalies flagged but retained.


In [10]:
cleaned.to_csv(f"{CLEANED_DIR}/cleaned_resale_flat_prices.csv", index=False)
print(f"Wrote {len(cleaned):,} rows -> {CLEANED_DIR}/cleaned_resale_flat_prices.csv")


Wrote 10,462 rows -> data/cleaned/cleaned_resale_flat_prices.csv


## Section 7 — Transformation: `Resale Identifier`

`S` + `BBB` (first 3 digits of `block`, zero-padded) + `PP` (first 2 digits of
the average resale price for that row's year-month/town/flat_type group) +
`MM` (month of the row) + `T` (first letter of town). Full construction logic
lives in `pipeline.add_resale_identifier`.

Per the requirement, duplicate records **on this new identifier** are then
resolved the same way (higher price wins) — this is on top of, not instead of,
the composite-key dedup in Section 5, since two structurally different rows
can coincidentally collide on the 8-character identifier.


In [11]:
transformed = pl.add_resale_identifier(cleaned)
transformed, failed_transform_dup = pl.dedup_after_transform(transformed)
print(f"Rows discarded for duplicate resale_identifier (lower price): {len(failed_transform_dup):,}")
transformed[["month", "town", "flat_type", "block", "resale_price", "resale_identifier"]].head()


Rows discarded for duplicate resale_identifier (lower price): 19
     month           town flat_type block  resale_price resale_identifier
0  2013-12        GEYLANG    3 ROOM   235      289000.0         S2353012G
1  2012-07        PUNGGOL    3 ROOM  395B      281000.0         S3952907P
2  2012-04  BUKIT PANJANG    2 ROOM   118      180500.0         S1181804B
3  2014-06    JURONG EAST    1 ROOM   768      148500.0         S7681406J
4  2013-06       CLEMENTI    5 ROOM   204      473000.0         S2044706C


In [12]:
transformed.to_csv(f"{TRANSFORMED_DIR}/transformed_resale_flat_prices.csv", index=False)
print(f"Wrote {len(transformed):,} rows -> {TRANSFORMED_DIR}/transformed_resale_flat_prices.csv")

failed_all = pd.concat([failed_validation, failed_dup, failed_transform_dup], ignore_index=True, sort=False)
failed_all.to_csv(f"{FAILED_DIR}/failed_records.csv", index=False)
print(f"Wrote {len(failed_all):,} rows -> {FAILED_DIR}/failed_records.csv")
failed_all["_fail_reason"].value_counts()


Wrote 10,443 rows -> data/transformed/transformed_resale_flat_prices.csv
Wrote 109 rows -> data/failed/failed_records.csv
_fail_reason
duplicate_key_lower_price                  52
invalid_flat_model;                        21
duplicate_resale_identifier_lower_price    19
invalid_storey_range;                      13
invalid_month;                              4


## Section 8 — Hashing

`resale_identifier` is hashed with **SHA-256**: a cryptographic, irreversible
hash function. It is used here (rather than e.g. MD5) because SHA-256's 256-bit
digest space makes accidental collisions across a dataset this size
negligibly unlikely, and unlike a reversible transform (encryption, simple
substitution), the original identifier cannot be recovered from the hash.
Uniqueness is preserved because SHA-256 is applied deterministically to the
full, untruncated identifier string (no bucketing), so two different
identifiers essentially never map to the same hash, and the same identifier
always maps to the same hash (needed for joins/dedup downstream). The
`hashed` output = `cleaned` (post-transform) data **+** the hash column.


In [13]:
hashed = pl.hash_identifier(transformed)
hashed.to_csv(f"{HASHED_DIR}/hashed_resale_flat_prices.csv", index=False)
print(f"Wrote {len(hashed):,} rows -> {HASHED_DIR}/hashed_resale_flat_prices.csv")
hashed[["resale_identifier", "resale_identifier_hash"]].head()


Wrote 10,443 rows -> data/hashed/hashed_resale_flat_prices.csv
  resale_identifier                                            resale_identifier_hash
0         S2353012G  550f18e2317be4de1c4b81b84b603da6335c28c5df2e23f8ace255f42b314e41
1         S3952907P  0ecd5b8a6f8e3335070e1632407ca9d939490d3bc97e2ff982f18762662a86ca
2         S1181804B  4ded6c57c26df57c16fefdd8fb213a4a6bab3b1a2bcf7e19ae7e5ae3ae0bcda7
3         S7681406J  4f4a8d137da33ec8ccd5ab339bbc4d944a14918bb70301907ba4fcf51e6be067
4         S2044706C  180c4dea4c62feabc3ef7fcc88591f916393016f65b4412e07ebcf56f4c8de83


## Summary


In [14]:
summary = pd.DataFrame({
    "stage": ["raw (combined)", "cleaned", "transformed", "failed (total)", "hashed"],
    "row_count": [len(master), len(cleaned), len(transformed), len(failed_all), len(hashed)],
})
summary


            stage  row_count
0  raw (combined)      10552
1         cleaned      10462
2     transformed      10443
3  failed (total)        109
4          hashed      10443


## Output structure

```
data/
├── raw/            # source files, untouched
├── cleaned/         # passed validation + dedup, anomalies flagged
├── transformed/     # + resale_identifier, post-transform dedup applied
├── failed/          # every discarded/invalid record, with _fail_reason
└── hashed/           # transformed data + resale_identifier_hash (SHA-256)
```

**Assumptions made (documented per assignment requirement 6 / general note):**
- Lease start = 1 Jan of `lease_commence_date` year; "today" = notebook run date.
- Validation thresholds (min support counts) are heuristic choices tuned to
  this dataset's size; on a much larger/smaller dataset they should be
  revisited.
- Price anomaly z-score threshold of 3 is a common statistical convention
  (~99.7% of a normal distribution falls within ±3σ) but is a heuristic, not
  a hard business rule — flagged rows are for review, not automatic exclusion.
- `_source_file`, `_fail_reason`, `_price_zscore`, `_is_price_anomaly`,
  `_month_parsed`, `_year_month` columns prefixed with `_` are pipeline
  metadata, not part of the original business schema.
